# LeetCode #16 — 3Sum Closest: Two Pointers để tìm tổng gần nhất

**Mức độ:** Medium · **Chủ đề:** Array, Two Pointers, Sorting

Skeleton trên LeetCode:

```python
class Solution:
    def threeSumClosest(self, nums: List[int], target: int) -> int:
```

Đây là "người anh em" của #15 3Sum. Thay vì tìm bộ ba có tổng **bằng** 0, ta tìm bộ ba có tổng **gần `target` nhất**.

Mục tiêu của notebook:

1. Dùng lại khung **sắp xếp + cố định một số + Two Pointers** từ bài 3Sum.
2. Hiểu cách **theo dõi đáp án tốt nhất** bằng khoảng cách `abs(total - target)`.
3. Hiểu vì sao hai con trỏ vẫn không bỏ sót khi điều kiện là "gần nhất" chứ không phải "bằng".

# 1. Bài toán là gì?

Cho mảng `nums` dài `n` và số `target`. Chọn **ba số ở ba vị trí khác nhau** sao cho tổng **gần `target` nhất**. Trả về **tổng** đó.

Đề đảm bảo mỗi input có **đúng một** đáp án.

**Constraints (theo đề):**

- `3 <= nums.length <= 500`
- `-1000 <= nums[i] <= 1000`
- `-10^4 <= target <= 10^4`

---

## Ví dụ 1

```text
nums = [-1, 2, 1, -4], target = 1  ->  2
```

Bộ ba `(-1, 2, 1)` có tổng `2`, cách `target` một khoảng `|2 - 1| = 1` — nhỏ nhất.

## Ví dụ 2

```text
nums = [0, 0, 0], target = 1  ->  0
```

---

## Điều quan trọng

Trả về **tổng** của bộ ba, không phải bộ ba, cũng không phải khoảng cách.

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Sắp xếp (sorting)
2. Hai con trỏ (two pointers)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Sắp xếp (sorting)

### Sắp xếp trong Python

- `sorted(a)` trả về **list mới**; `a.sort()` sắp xếp **tại chỗ** (trả về `None`).
- `key=` chọn tiêu chí: `sorted(words, key=len)`, `sorted(pairs, key=lambda p: p[1])`.
- `reverse=True` để giảm dần; sắp xếp theo nhiều tiêu chí bằng `key` trả về tuple.
- Thuật toán bên trong là **Timsort**: `O(n log n)`, **ổn định** (stable — hai phần tử bằng nhau giữ nguyên thứ tự ban đầu).

In [1]:
a_sort = [5, 2, 9, 1, 5, 6]
print("sorted:", sorted(a_sort), "| gốc không đổi:", a_sort)
a_sort.sort(reverse=True)
print("sort giảm dần tại chỗ:", a_sort)

nguoi = [("An", 25), ("Bình", 20), ("Chi", 25), ("Dũng", 20)]
print("theo tuổi (ổn định — An trước Chi):", sorted(nguoi, key=lambda p: p[1]))
print("tuổi giảm, rồi tên tăng:", sorted(nguoi, key=lambda p: (-p[1], p[0])))
print("chuỗi sắp theo ký tự:", "".join(sorted("banana")))

sorted: [1, 2, 5, 5, 6, 9] | gốc không đổi: [5, 2, 9, 1, 5, 6]
sort giảm dần tại chỗ: [9, 6, 5, 5, 2, 1]
theo tuổi (ổn định — An trước Chi): [('Bình', 20), ('Dũng', 20), ('An', 25), ('Chi', 25)]
tuổi giảm, rồi tên tăng: [('An', 25), ('Chi', 25), ('Bình', 20), ('Dũng', 20)]
chuỗi sắp theo ký tự: aaabnn


### Vì sao sắp xếp hay là bước đầu tiên?

Sau khi sắp xếp: phần tử **bằng nhau đứng cạnh nhau** (dễ bỏ trùng), có thể dùng **hai con trỏ** / **tìm nhị phân**, các khoảng chồng nhau **liền kề** nhau. Đổi lại tốn `O(n log n)`.

### Sắp xếp chèn (insertion sort)

Giống xếp bài trên tay: lấy lần lượt từng phần tử, **chèn** vào đúng vị trí trong phần đã sắp xếp bên trái. `O(n²)` xấu nhất, nhưng `O(n)` khi dữ liệu gần như đã sắp xếp.

In [2]:
def sap_xep_chen(a):
    a = a[:]
    for i in range(1, len(a)):
        x, j = a[i], i - 1
        while j >= 0 and a[j] > x:
            a[j + 1] = a[j]            # dịch phần tử lớn hơn sang phải
            j -= 1
        a[j + 1] = x
        print(f"   sau khi chèn {x}: {a[:i + 1]} | {a[i + 1:]}")
    return a

print("→", sap_xep_chen([5, 2, 4, 6, 1, 3]))

   sau khi chèn 2: [2, 5] | [4, 6, 1, 3]
   sau khi chèn 4: [2, 4, 5] | [6, 1, 3]
   sau khi chèn 6: [2, 4, 5, 6] | [1, 3]
   sau khi chèn 1: [1, 2, 4, 5, 6] | [3]
   sau khi chèn 3: [1, 2, 3, 4, 5, 6] | []
→ [1, 2, 3, 4, 5, 6]


### Sắp xếp trộn (merge sort) và phép trộn hai dãy đã sắp xếp

**Chia để trị**: chia đôi, sắp xếp mỗi nửa (đệ quy), rồi **trộn** hai nửa đã sắp xếp. Trộn: hai con trỏ ở đầu hai dãy, mỗi bước lấy phần tử nhỏ hơn. `O(n log n)` mọi trường hợp, ổn định (nhờ lấy bên trái khi bằng nhau).

```text
[5,2,4,6,1,3] → [5,2,4] [6,1,3] → [5] [2,4] [6] [1,3] → … → [2,4,5] [1,3,6] → [1,2,3,4,5,6]
```

In [3]:
def tron(x, y):
    i = j = 0
    out = []
    while i < len(x) and j < len(y):
        if x[i] <= y[j]:               # <= giữ ổn định
            out.append(x[i]); i += 1
        else:
            out.append(y[j]); j += 1
    out.extend(x[i:]); out.extend(y[j:])
    return out

def sap_xep_tron(a, sau=0):
    if len(a) <= 1:
        return a
    mid = len(a) // 2
    trai, phai = sap_xep_tron(a[:mid], sau + 1), sap_xep_tron(a[mid:], sau + 1)
    kq = tron(trai, phai)
    print("    " * sau + f"trộn {trai} + {phai} = {kq}")
    return kq

print("→", sap_xep_tron([5, 2, 4, 6, 1, 3]))

        trộn [2] + [4] = [2, 4]
    trộn [5] + [2, 4] = [2, 4, 5]
        trộn [1] + [3] = [1, 3]
    trộn [6] + [1, 3] = [1, 3, 6]
trộn [2, 4, 5] + [1, 3, 6] = [1, 2, 3, 4, 5, 6]
→ [1, 2, 3, 4, 5, 6]


### Bảng so sánh

| Thuật toán | Tốt nhất | Xấu nhất | Bộ nhớ phụ | Ổn định |
|---|---|---|---|---|
| Insertion sort | `O(n)` | `O(n²)` | `O(1)` | có |
| Merge sort | `O(n log n)` | `O(n log n)` | `O(n)` (mảng) | có |
| Quick sort | `O(n log n)` | `O(n²)` | `O(log n)` | không |
| Timsort (`sorted`) | `O(n)` | `O(n log n)` | `O(n)` | có |

Lỗi hay gặp: `a = a.sort()` (gán `None`); quên rằng `sorted` tạo list mới (tốn bộ nhớ); `key` trả về kiểu không so sánh được.

## 2.2 Hai con trỏ (two pointers)

### Ý tưởng

Dùng **hai chỉ số** di chuyển trên mảng/chuỗi theo quy tắc, để thay vòng lặp lồng `O(n²)` bằng một lượt `O(n)`. Hai kiểu chính:

**1. Hai đầu tiến vào nhau** (thường trên mảng đã sắp xếp):

```text
[1, 2, 4, 7, 11, 15], tìm hai số có tổng 15
 L                 R     1 + 15 = 16 > 15 → R lùi (cần nhỏ hơn)
 L             R         1 + 11 = 12 < 15 → L tiến (cần lớn hơn)
    L          R         2 + 11 = 13 < 15 → L tiến
       L       R         4 + 11 = 15 ✓
```

Vì sao được phép bỏ? Khi `a[L] + a[R] > target`, mọi cặp `(L', R)` với `L' > L` còn lớn hơn → `R` không thể là đáp án, bỏ `R`.

**2. Cùng chiều — con trỏ đọc và con trỏ ghi** (sửa mảng tại chỗ):

```text
xóa mọi số 0, giữ thứ tự: [0, 1, 0, 3, 12]
read đi qua từng phần tử; gặp số khác 0 thì chép vào vị trí write, write += 1
→ [1, 3, 12, ...]
```

In [4]:
def tong_hai_so_da_sap_xep(a, target):
    L, R = 0, len(a) - 1
    while L < R:
        s = a[L] + a[R]
        print(f"   L={L} R={R}: {a[L]} + {a[R]} = {s}")
        if s == target:
            return L, R
        if s < target:
            L += 1
        else:
            R -= 1
    return None

print("→", tong_hai_so_da_sap_xep([1, 2, 4, 7, 11, 15], 15))

def bo_so_khong(a):
    write = 0
    for read in range(len(a)):
        if a[read] != 0:
            a[write] = a[read]
            write += 1
    return a[:write]

print("bỏ số 0:", bo_so_khong([0, 1, 0, 3, 12]))

def la_palindrome(s):
    i, j = 0, len(s) - 1
    while i < j:
        if s[i] != s[j]:
            return False
        i += 1; j -= 1
    return True
print("'racecar' palindrome?", la_palindrome("racecar"), "| 'abca'?", la_palindrome("abca"))

   L=0 R=5: 1 + 15 = 16
   L=0 R=4: 1 + 11 = 12
   L=1 R=4: 2 + 11 = 13
   L=2 R=4: 4 + 11 = 15
→ (2, 4)
bỏ số 0: [1, 3, 12]
'racecar' palindrome? True | 'abca'? False


### Đảo ngược một đoạn tại chỗ

Hai con trỏ ở hai đầu, đổi chỗ rồi tiến vào trong — dùng rất nhiều (đảo mảng, xoay mảng, đảo từ).

In [5]:
def dao_doan(a, i, j):
    while i < j:
        a[i], a[j] = a[j], a[i]
        i += 1; j -= 1

arr_tp = [1, 2, 3, 4, 5, 6]
dao_doan(arr_tp, 1, 4)
print("đảo đoạn [1..4]:", arr_tp)

đảo đoạn [1..4]: [1, 5, 4, 3, 2, 6]


### Lỗi hay gặp

- Quên **sắp xếp** trước khi dùng hai đầu cho bài tổng.
- Điều kiện `L <= R` thay vì `L < R` → dùng một phần tử hai lần.
- Không chứng minh được vì sao bỏ một đầu là an toàn → dễ bỏ mất đáp án.
- Con trỏ ghi vượt con trỏ đọc (không thể xảy ra nếu `write` chỉ tăng khi ghi).

## 2.3 Áp dụng vào bài này

Giống 3Sum: **sắp xếp**, cố định một số, dùng **hai con trỏ** cho hai số còn lại. Tổng nhỏ hơn target thì tăng con trỏ trái, lớn hơn thì giảm con trỏ phải; mỗi bước cập nhật tổng có `abs(tổng − target)` nhỏ nhất.

# 3. Nền tảng: đo "độ gần" bằng `abs`

Khoảng cách giữa `total` và `target` là `abs(total - target)`. Tổng nào có khoảng cách nhỏ hơn thì "gần hơn".

In [6]:
target = 1
for total in [2, -1, 1, 5]:
    print(f"total = {total:>2}, khoảng cách = abs({total} - {target}) = {abs(total - target)}")

total =  2, khoảng cách = abs(2 - 1) = 1
total = -1, khoảng cách = abs(-1 - 1) = 2
total =  1, khoảng cách = abs(1 - 1) = 0
total =  5, khoảng cách = abs(5 - 1) = 4


Mẫu "cập nhật đáp án tốt nhất":

```python
best = None
for total in ...:
    if best is None or abs(total - target) < abs(best - target):
        best = total
```

Mẹo: khởi tạo `best` bằng **một tổng có thật** (ví dụ tổng ba phần tử đầu) để tránh `None`.

In [7]:
candidates = [5, -3, 2, 0, 4]
target = 1
best = candidates[0]
for total in candidates[1:]:
    if abs(total - target) < abs(best - target):
        best = total
print("Tổng gần", target, "nhất:", best)

Tổng gần 1 nhất: 2


# 4. Cách dễ nghĩ nhất: thử mọi bộ ba

In [8]:
from typing import List

def three_sum_closest_bruteforce(nums: List[int], target: int) -> int:
    n = len(nums)
    best = nums[0] + nums[1] + nums[2]
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                total = nums[i] + nums[j] + nums[k]
                if abs(total - target) < abs(best - target):
                    best = total
    return best

print(three_sum_closest_bruteforce([-1, 2, 1, -4], 1))
print(three_sum_closest_bruteforce([0, 0, 0], 1))

2
0


`O(n³)`: với `n = 500` khoảng `2 * 10^7` bộ ba — Python chạy được nhưng chậm, và trong phỏng vấn cần `O(n²)`.

# 5. Ý tưởng chính: sắp xếp + Two Pointers

Giống 3Sum: sắp xếp, cố định `nums[i]`, đặt `left = i + 1`, `right = n - 1`.

Mỗi bước tính `total = nums[i] + nums[left] + nums[right]`, **cập nhật `best`**, rồi:

- `total < target` → tổng cần **lớn hơn** → `left += 1`.
- `total > target` → tổng cần **nhỏ hơn** → `right -= 1`.
- `total == target` → khoảng cách bằng 0, **không thể tốt hơn** → trả về ngay.

## Vì sao không bỏ sót?

Giả sử `total < target`. Mọi cặp `(left, k)` với `left < k < right` có tổng **nhỏ hơn hoặc bằng** `total` (mảng tăng dần), tức còn **xa target hơn hoặc bằng** (cùng nằm bên dưới target). Vậy phần tử `left` không giúp được gì thêm → bỏ `left` an toàn.

Lập luận tương tự khi `total > target`.

In [9]:
nums = sorted([-1, 2, 1, -4])
target = 1
i = 0
left, right = 1, len(nums) - 1
total = nums[i] + nums[left] + nums[right]
print("nums sắp xếp:", nums)
print(f"i={i}, left={left}, right={right}, total={total} < target={target}")
print("Các tổng (i, left, k) với k bên trong:", [nums[i] + nums[left] + nums[k] for k in range(left + 1, right)])
print("-> đều <= total, nên còn xa target hơn: bỏ left là an toàn")

nums sắp xếp: [-4, -1, 1, 2]
i=0, left=1, right=3, total=-3 < target=1
Các tổng (i, left, k) với k bên trong: [-4]
-> đều <= total, nên còn xa target hơn: bỏ left là an toàn


# 6. Lời giải chính

In [10]:
def three_sum_closest(nums: List[int], target: int) -> int:
    nums = sorted(nums)
    n = len(nums)
    best = nums[0] + nums[1] + nums[2]

    for i in range(n - 2):
        left, right = i + 1, n - 1
        while left < right:
            total = nums[i] + nums[left] + nums[right]

            if abs(total - target) < abs(best - target):
                best = total

            if total < target:
                left += 1
            elif total > target:
                right -= 1
            else:
                return total            # khoảng cách 0: tốt nhất có thể

    return best

print(three_sum_closest([-1, 2, 1, -4], 1))
print(three_sum_closest([0, 0, 0], 1))

2
0


# 7. Mô phỏng từng bước

`nums = [-1, 2, 1, -4]` → sắp xếp `[-4, -1, 1, 2]`, `target = 1`.

| i (giá trị) | left | right | total | khoảng cách | best | Dời |
|---|---|---|---|---|---|---|
| 0 (-4) | 1 (-1) | 3 (2) | -3 | 4 | -3 | left (< 1) |
| 0 (-4) | 2 (1) | 3 (2) | -1 | 2 | -1 | left → dừng |
| 1 (-1) | 2 (1) | 3 (2) | **2** | **1** | **2** | right (> 1) → dừng |

Kết quả `2`. (`best` khởi tạo là `-4 + -1 + 1 = -4`, khoảng cách 5, bị thay ngay ở bước đầu.)

In [11]:
def trace_three_sum_closest(nums: List[int], target: int) -> int:
    nums = sorted(nums)
    n = len(nums)
    best = nums[0] + nums[1] + nums[2]
    print(f"Sắp xếp: {nums}, target = {target}, best ban đầu = {best}")
    for i in range(n - 2):
        left, right = i + 1, n - 1
        while left < right:
            total = nums[i] + nums[left] + nums[right]
            updated = abs(total - target) < abs(best - target)
            if updated:
                best = total
            move = "left += 1" if total < target else ("right -= 1" if total > target else "BẰNG target -> trả về")
            print(f"i={i}({nums[i]:>2}) left={left}({nums[left]:>2}) right={right}({nums[right]:>2}) "
                  f"total={total:>3} |total-target|={abs(total - target):>2} best={best:>3}"
                  f"{' (cập nhật)' if updated else '           '} -> {move}")
            if total < target:
                left += 1
            elif total > target:
                right -= 1
            else:
                return total
    return best

print("Đáp án:", trace_three_sum_closest([-1, 2, 1, -4], 1))

Sắp xếp: [-4, -1, 1, 2], target = 1, best ban đầu = -4
i=0(-4) left=1(-1) right=3( 2) total= -3 |total-target|= 4 best= -3 (cập nhật) -> left += 1
i=0(-4) left=2( 1) right=3( 2) total= -1 |total-target|= 2 best= -1 (cập nhật) -> left += 1
i=1(-1) left=2( 1) right=3( 2) total=  2 |total-target|= 1 best=  2 (cập nhật) -> right -= 1
Đáp án: 2


# 8. Phiên bản đúng format LeetCode

In [12]:
class Solution:
    def threeSumClosest(self, nums: List[int], target: int) -> int:
        nums.sort()
        n = len(nums)
        best = nums[0] + nums[1] + nums[2]
        for i in range(n - 2):
            left, right = i + 1, n - 1
            while left < right:
                total = nums[i] + nums[left] + nums[right]
                if abs(total - target) < abs(best - target):
                    best = total
                if total < target:
                    left += 1
                elif total > target:
                    right -= 1
                else:
                    return total
        return best

In [13]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<30} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.threeSumClosest([-1, 2, 1, -4], 1), 2)
check("Ví dụ 2", sol.threeSumClosest([0, 0, 0], 1), 0)
check("Đúng 3 phần tử", sol.threeSumClosest([1, 1, 1], 100), 3)
check("Có tổng đúng bằng target", sol.threeSumClosest([1, 2, 3, 4], 9), 9)
check("Toàn số âm", sol.threeSumClosest([-5, -4, -3, -2], -100), -12)
check("Target nhỏ hơn mọi tổng", sol.threeSumClosest([4, 0, 5, -5, 3, 3, 0, -4, -5], -2), -2)

Ví dụ 1                        PASS  2
Ví dụ 2                        PASS  0
Đúng 3 phần tử                 PASS  3
Có tổng đúng bằng target       PASS  9
Toàn số âm                     PASS  -12
Target nhỏ hơn mọi tổng        PASS  -2


## Đối chiếu với Brute Force

Đề đảm bảo đáp án **duy nhất**, nên khi sinh test ngẫu nhiên ta chỉ giữ những input mà tổng gần nhất là duy nhất (không có hai tổng khác nhau cùng khoảng cách, như `target - 1` và `target + 1`).

In [14]:
import random
from itertools import combinations

random.seed(16)
tested = 0
while tested < 1500:
    nums = [random.randint(-10, 10) for _ in range(random.randint(3, 10))]
    target = random.randint(-25, 25)
    sums = {sum(c) for c in combinations(nums, 3)}
    d = min(abs(s - target) for s in sums)
    if sum(1 for s in sums if abs(s - target) == d) > 1:
        continue                         # không thỏa "đáp án duy nhất"
    assert sol.threeSumClosest(nums[:], target) == three_sum_closest_bruteforce(nums, target), (nums, target)
    tested += 1
print("1500 test ngẫu nhiên (đáp án duy nhất): khớp Brute Force")

1500 test ngẫu nhiên (đáp án duy nhất): khớp Brute Force


# 9. Vì sao đúng? Độ phức tạp?

**Đúng:** với mỗi `i`, hai con trỏ chỉ bỏ những phần tử mà mọi tổng còn lại dùng nó đều **xa target hơn hoặc bằng** tổng vừa xét (mục 5). Tổng vừa xét đã được so với `best`. Do đó tổng tốt nhất với mỗi `i` cố định đều được gặp, và `best` là tốt nhất trên mọi `i`.

**Độ phức tạp:**

```text
Time:  O(n log n) sắp xếp + O(n²) hai con trỏ = O(n²)
Space: O(1) bộ nhớ phụ (không tính sắp xếp)
```

| Cách làm | Thời gian | Bộ nhớ |
|---|---|---|
| Brute force | O(n³) | O(1) |
| Sắp xếp + Two Pointers | O(n²) | O(1) |

Khác 3Sum: **không cần bỏ trùng** vì ta chỉ trả về một con số. (Bỏ trùng `nums[i] == nums[i-1]` vẫn được phép như một tối ưu nhỏ.)

# 10. Những lỗi dễ mắc

## Lỗi 1: Khởi tạo `best` bằng một giá trị "giả"

`best` phải là **tổng của một bộ ba có thật**. Nếu khởi tạo `best = 0` (hay một số tùy ý), và giá trị giả đó tình cờ gần `target` hơn mọi tổng thật, hàm sẽ trả về một con số không phải tổng của bộ ba nào:

In [15]:
def wrong_init_zero(nums: List[int], target: int) -> int:
    nums = sorted(nums)
    best = 0                                   # SAI: 0 có thể không phải tổng của bộ ba nào
    for i in range(len(nums) - 2):
        left, right = i + 1, len(nums) - 1
        while left < right:
            total = nums[i] + nums[left] + nums[right]
            if abs(total - target) < abs(best - target):
                best = total
            if total < target:
                left += 1
            elif total > target:
                right -= 1
            else:
                return total
    return best

print("Sai :", wrong_init_zero([1, 2, 3], 1))
print("Đúng:", three_sum_closest([1, 2, 3], 1))

Sai :

 0
Đúng: 6


## Lỗi 2: So sánh khoảng cách mà không dùng `abs`

`total - target < best - target` chỉ đơn giản là `total < best` — tìm tổng **nhỏ nhất**, không phải gần nhất.

## Lỗi 3: Trả về khoảng cách thay vì tổng

Đề hỏi **tổng**. Trả `abs(best - target)` là sai.

## Lỗi 4: Quên sắp xếp

Two Pointers chỉ đúng trên mảng đã sắp xếp; thiếu `sort` thì quy tắc dời con trỏ vô nghĩa.

# 11. Mẹo nhận diện pattern

**"Tối ưu hóa gần nhất" + Two Pointers:**

- Khung giống hệt bài tìm tổng chính xác, chỉ thêm một dòng **cập nhật best** trước khi dời con trỏ.
- Nếu gặp tổng đúng bằng target, **dừng sớm**.

Mẫu dùng lại ở: **#16 3Sum Closest**, **#259 3Sum Smaller**, **#658 Find K Closest Elements**, "Two Sum Closest" (tìm cặp gần target nhất).

## Template

```python
nums.sort()
best = nums[0] + nums[1] + nums[2]
for i in range(n - 2):
    left, right = i + 1, n - 1
    while left < right:
        total = nums[i] + nums[left] + nums[right]
        if abs(total - target) < abs(best - target):
            best = total
        if total < target:   left += 1
        elif total > target: right -= 1
        else:                return total
return best
```

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Trace cho `nums = [1, 1, -1, -1, 3]`, `target = -1`. Kết quả là gì?

## Bài 2 — Đoán output

```python
print(abs(-3 - 1), abs(2 - 1))
print(sorted([4, -1, 0, 2])[0:3], sum(sorted([4, -1, 0, 2])[0:3]))
print(min([5, -3, 2], key=lambda t: abs(t - 1)))
```

## Bài 3 — Two Sum Closest

Cho mảng `nums` (độ dài ≥ 2) và `target`. Trả về **tổng hai số** gần `target` nhất. Yêu cầu `O(n log n)`.

```python
two_sum_closest([1, 4, 7, 10], 15) -> 14     # 4 + 10 (mỗi phần tử chỉ dùng một lần)
two_sum_closest([-5, 2, 9], 0)     -> -3
```

In [16]:
def exercise_3_two_sum_closest(nums: List[int], target: int) -> int:
    # TODO: sắp xếp, hai con trỏ, cập nhật best
    pass


# assert exercise_3_two_sum_closest([1, 4, 7, 10], 15) == 14
# assert exercise_3_two_sum_closest([-5, 2, 9], 0) == -3
# assert exercise_3_two_sum_closest([1, 2], 100) == 3

## Bài 4 — Điền chỗ trống: lời giải chính

In [17]:
def exercise_4_three_sum_closest(nums: List[int], target: int) -> int:
    nums = sorted(nums)
    n = len(nums)
    # TODO 1: khởi tạo best bằng một tổng có thật
    for i in range(n - 2):
        left, right = i + 1, n - 1
        while left < right:
            # TODO 2: tính total, cập nhật best nếu gần hơn
            # TODO 3: dời con trỏ, hoặc trả về ngay nếu total == target
            pass
    return best


# assert exercise_4_three_sum_closest([-1, 2, 1, -4], 1) == 2
# assert exercise_4_three_sum_closest([0, 0, 0], 1) == 0

## Bài 5 — Mini challenge: trả về luôn bộ ba

Trả về `(tổng, bộ_ba)` với `bộ_ba` là list ba giá trị đã sắp xếp tăng dần tạo ra tổng gần nhất.

```python
closest_triplet([-1, 2, 1, -4], 1) -> (2, [-1, 1, 2])
```

Sau đó, hãy thử thêm: nếu `nums` có tới `10^4` phần tử thì `O(n²)` còn chạy kịp không trong Python? (Ước lượng số phép tính.)

In [18]:
def exercise_5_closest_triplet(nums: List[int], target: int):
    # TODO
    pass


# assert exercise_5_closest_triplet([-1, 2, 1, -4], 1) == (2, [-1, 1, 2])
# assert exercise_5_closest_triplet([0, 0, 0], 1) == (0, [0, 0, 0])

# 13. Gợi ý

**Bài 1**
- Mức 1: sắp xếp: `[-1, -1, 1, 1, 3]`.
- Mức 2: tổng nhỏ nhất có thể là `-1 + -1 + 1 = -1`.
- Mức 3: khi gặp đúng target, thuật toán trả về ngay.

**Bài 3**
- Mức 1: giống lời giải chính nhưng không có vòng `i`.
- Mức 2: `best = nums[0] + nums[1]` sau khi sắp xếp.

**Bài 4**
- Mức 1: `best = nums[0] + nums[1] + nums[2]`.
- Mức 2: cập nhật best **trước** khi dời con trỏ.

**Bài 5**
- Mức 1: lưu thêm `best_triplet` mỗi khi cập nhật `best`.
- Mức 2: `[nums[i], nums[left], nums[right]]` đã tăng dần vì mảng đã sắp xếp và `i < left < right`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [19]:
print("Đáp án:", trace_three_sum_closest([1, 1, -1, -1, 3], -1))

Sắp xếp: [-1, -1, 1, 1, 3], target = -1, best ban đầu = -1
i=0(-1) left=1(-1) right=4( 3) total=  1 |total-target|= 2 best= -1            -> right -= 1
i=0(-1) left=1(-1) right=3( 1) total= -1 |total-target|= 0 best= -1            -> BẰNG target -> trả về
Đáp án: -1


## Đáp án Bài 2

In [20]:
print(abs(-3 - 1), abs(2 - 1))                                    # 4 1
print(sorted([4, -1, 0, 2])[0:3], sum(sorted([4, -1, 0, 2])[0:3])) # [-1, 0, 2] 1
print(min([5, -3, 2], key=lambda t: abs(t - 1)))                  # 2

4 1
[-1, 0, 2] 1
2


`min(..., key=...)` là cách viết gọn của mẫu "tìm phần tử tốt nhất theo một tiêu chí".

## Đáp án Bài 3

In [21]:
def answer_3_two_sum_closest(nums: List[int], target: int) -> int:
    nums = sorted(nums)
    left, right = 0, len(nums) - 1
    best = nums[left] + nums[right]
    while left < right:
        total = nums[left] + nums[right]
        if abs(total - target) < abs(best - target):
            best = total
        if total < target:
            left += 1
        elif total > target:
            right -= 1
        else:
            return total
    return best

assert answer_3_two_sum_closest([1, 4, 7, 10], 15) == 14
assert answer_3_two_sum_closest([-5, 2, 9], 0) == -3
assert answer_3_two_sum_closest([1, 2], 100) == 3
print("Bài 3 PASS")

Bài 3 PASS


Với `[1, 4, 7, 10]` và `target = 15`: các tổng có thể là 5, 8, 11, 11, 14, 17. Hai ứng viên gần nhất là `14` (cách 1) và `17` (cách 2) → `14`.

## Đáp án Bài 4

In [22]:
def answer_4_three_sum_closest(nums: List[int], target: int) -> int:
    nums = sorted(nums)
    n = len(nums)
    best = nums[0] + nums[1] + nums[2]
    for i in range(n - 2):
        left, right = i + 1, n - 1
        while left < right:
            total = nums[i] + nums[left] + nums[right]
            if abs(total - target) < abs(best - target):
                best = total
            if total < target:
                left += 1
            elif total > target:
                right -= 1
            else:
                return total
    return best

assert answer_4_three_sum_closest([-1, 2, 1, -4], 1) == 2
assert answer_4_three_sum_closest([0, 0, 0], 1) == 0
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [23]:
def answer_5_closest_triplet(nums: List[int], target: int):
    nums = sorted(nums)
    n = len(nums)
    best = nums[0] + nums[1] + nums[2]
    best_triplet = nums[:3]
    for i in range(n - 2):
        left, right = i + 1, n - 1
        while left < right:
            total = nums[i] + nums[left] + nums[right]
            if abs(total - target) < abs(best - target):
                best, best_triplet = total, [nums[i], nums[left], nums[right]]
            if total < target:
                left += 1
            elif total > target:
                right -= 1
            else:
                return total, [nums[i], nums[left], nums[right]]
    return best, best_triplet

assert answer_5_closest_triplet([-1, 2, 1, -4], 1) == (2, [-1, 1, 2])
assert answer_5_closest_triplet([0, 0, 0], 1) == (0, [0, 0, 0])
print("Bài 5 PASS")

Bài 5 PASS


Câu hỏi thêm: với `n = 10^4`, `n² / 2 = 5 * 10^7` bước. Mỗi bước Python mất cỡ vài chục nano giây tới 0.1 micro giây → khoảng vài giây. Chạy được nhưng chậm; ngôn ngữ biên dịch (C++/Java) sẽ nhanh hơn nhiều lần.

# 15. Tổng kết

> Khung giống 3Sum. Khác biệt duy nhất: thêm dòng **cập nhật `best` theo `abs(total - target)`**, và **trả về ngay** nếu khoảng cách bằng 0.

---

## Checklist trước khi nộp

- [ ] Đã sắp xếp?
- [ ] Khởi tạo `best` bằng một **tổng có thật**?
- [ ] So sánh bằng `abs(total - target)`?
- [ ] Cập nhật `best` **trước** khi dời con trỏ?
- [ ] Trả về **tổng**, không phải khoảng cách?